# 01b – Extra training areas (round 2)

Round 1 (notebook 01, plan v2) showed that **herbaceous wetland** (0.5% of training pixels, test IoU 0.01)
and **bare / sparse vegetation** (2.5%, test IoU 0.06) are the weakest classes, and that the model overfits
(train accuracy 0.78 vs validation 0.58). Both point to a **data** problem, so round 2 adds 34 new
**training** areas: 12 chosen for deserts and wetlands, 22 for more cities, farmland, forest and steppe.

Validation and test areas stay exactly the same as in round 1, so both rounds can be compared fairly.
The new patches are saved separately in `data/patches_v3_extra/` (train only); the training notebook
combines them with the round-1 patches.

Everything else (track choice, coverage rule, export, file checks, 7 classes) is identical to notebook 01.

# Original description of the steps

Input: Sentinel-1 GRD (VV, VH), monthly median of one satellite track.
Labels: ESA WorldCover 2021, on the same 10 m grid as the radar.

Steps:
1. Setup (packages and Earth Engine)
2. Choose the areas and assign train / validation / test **by area**
3. Check which satellite track covers each area in all four months
4. Check coverage and class distribution
5. Export radar (4 months) and labels for every area as GeoTIFF
6. Cut everything into 256 × 256 patches and save them as NumPy arrays

# 1. Setup

The JupyterHub resets packages installed with `pip --user` when the server restarts, so this cell
installs the requirements again if they are missing. It only takes a few seconds when they are already there.

In [1]:
import sys
import subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    # Kaggle already has NumPy, requests and matplotlib; only add what is missing
    packages = ["earthengine-api==1.7.49", "rasterio==1.4.4"]
else:
    packages = ["-r", "requirements.txt"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("packages ready")

packages ready


In [2]:
from pathlib import Path
import json
import math

import numpy as np
import requests
import rasterio
import ee

PROJECT_ID = "sentinel1-lulc-project-512618"

# Only asks for a new login when the saved token is missing (e.g. after a hub restart).
# Clear the output of this cell before sharing the notebook, it shows a one-time code.
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine ready:", ee.Number(1).getInfo())

Earth Engine ready: 1


# 2. Areas and split

Every area goes completely into one split (train, validation or test), so neighbouring pixels
of the same place never end up in both training and testing.

Changes compared to version 1:
- **Mannheim removed.** It is the case-study city for the prototype and the screen recording, so the model must not see it during training.
- **11 cities added to training.** Version 1 had almost no built-up area in training except Mannheim.
- **Naivasha replaced by Nairobi.** Naivasha was 89% lake.
- **Vancouver replaced by Portland.** Only about 41% of Vancouver was covered by one track.
- **Bigger areas** (`BUFFER_METRES = 7700`, about 15.4 × 15.4 km). Each area gives 6 × 6 = 36 patches per month instead of 3 × 3.

In [3]:
YEAR = 2021
MONTHS = [1, 4, 7, 10]
BUFFER_METRES = 7700
PLAN_VERSION = "v3_extra"

# Round-1 areas (plan v2), only used here for the separation check
existing_locations = [
    # Europe
    ("Abisko",           "Europe",          18.830,  68.350, "train"),
    ("Lyon",             "Europe",           4.840,  45.760, "train"),
    ("Madrid",           "Europe",          -3.700,  40.420, "train"),
    ("Warsaw",           "Europe",          21.010,  52.230, "train"),
    ("Prague",           "Europe",          14.440,  50.080, "validation"),
    ("Toulouse",         "Europe",           1.444,  43.605, "test"),

    # Asia
    ("Ludhiana_rural",   "Asia",            75.650,  30.900, "train"),
    ("Sundarbans",       "Asia",            89.550,  22.100, "train"),
    ("Pune",             "Asia",            73.857,  18.520, "train"),
    ("Delhi",            "Asia",            77.210,  28.610, "train"),
    ("Bangkok_edge",     "Asia",           100.750,  13.850, "validation"),
    ("Hanoi_edge",       "Asia",           105.900,  21.100, "test"),

    # Africa
    ("Nairobi",          "Africa",          36.820,  -1.290, "train"),
    ("Marrakech_edge",   "Africa",          -8.150,  31.650, "train"),
    ("Cairo",            "Africa",          31.240,  30.040, "train"),
    ("Lagos",            "Africa",           3.380,   6.520, "train"),
    ("Arusha_edge",      "Africa",          36.750,  -3.450, "validation"),
    ("Pretoria_edge",    "Africa",          28.350, -25.750, "test"),

    # North America
    ("Sacramento_rural", "North America", -121.650,  38.550, "train"),
    ("Ontario_forest",   "North America",  -78.500,  45.500, "train"),
    ("Chicago",          "North America",  -87.750,  41.850, "train"),
    ("Mexico_City",      "North America",  -99.130,  19.430, "train"),
    ("Austin_edge",      "North America",  -97.600,  30.250, "validation"),
    ("Portland_edge",    "North America", -122.680,  45.520, "test"),

    # South America
    ("Santarem_edge",    "South America",  -54.800,  -2.550, "train"),
    ("Rosario_rural",    "South America",  -60.900, -33.000, "train"),
    ("Sao_Paulo",        "South America",  -46.630, -23.550, "train"),
    ("Curitiba_edge",    "South America",  -49.150, -25.550, "validation"),
    ("Cordoba_edge",     "South America",  -64.050, -31.450, "test"),

    # Oceania
    ("Wagga_rural",      "Oceania",        147.200, -35.100, "train"),
    ("Darwin_wetlands",  "Oceania",        131.300, -12.650, "train"),
    ("Melbourne",        "Oceania",        144.960, -37.810, "train"),
    ("Perth_edge",       "Oceania",        115.950, -31.800, "validation"),
    ("Christchurch",     "Oceania",        172.650, -43.500, "test"),
]

existing_areas = [
    {"name": name, "continent": continent, "lon": lon, "lat": lat, "split": split}
    for name, continent, lon, lat, split in existing_locations
]

# Round 2: new TRAINING areas only. Name, continent, longitude, latitude
new_locations = [
    # targeted: bare / desert
    ("Tabernas_desert",    "Europe",          -2.390,  37.050),
    ("Riyadh_edge",        "Asia",            46.450,  24.750),
    ("Ouarzazate",         "Africa",          -6.900,  30.920),
    ("Phoenix_edge",       "North America", -112.300,  33.350),
    ("Antofagasta",        "South America",  -70.400, -23.650),
    ("Alice_Springs",      "Oceania",        133.880, -23.700),
    # targeted: herbaceous wetland
    ("Camargue",           "Europe",           4.550,  43.550),
    ("Danube_delta",       "Europe",          29.300,  45.150),
    ("Poyang_lake",        "Asia",           116.250,  29.000),
    ("Okavango",           "Africa",          22.900, -19.300),
    ("Everglades_edge",    "North America",  -80.600,  25.800),
    ("Pantanal",           "South America",  -57.000, -17.600),
    # more diversity: cities, farmland, forest, steppe
    ("Dublin",             "Europe",          -6.260,  53.350),
    ("Po_valley",          "Europe",          10.500,  45.100),
    ("Jyvaskyla_forest",   "Europe",          25.700,  62.000),
    ("Andalusia_farmland", "Europe",          -4.800,  37.550),
    ("Jakarta",            "Asia",           106.850,  -6.200),
    ("Dhaka",              "Asia",            90.410,  23.810),
    ("Tehran",             "Asia",            51.400,  35.700),
    ("Astana_steppe",      "Asia",            71.450,  51.150),
    ("Mekong_delta",       "Asia",           105.800,  10.000),
    ("Addis_Ababa",        "Africa",          38.740,   9.030),
    ("Kano_rural",         "Africa",           8.500,  12.000),
    ("Kinshasa",           "Africa",          15.310,  -4.320),
    ("Nile_delta",         "Africa",          31.000,  30.800),
    ("Iowa_farmland",      "North America",  -93.600,  42.000),
    ("Alberta_prairie",    "North America", -113.500,  52.000),
    ("Guadalajara",        "North America", -103.350,  20.670),
    ("Bogota",             "South America",  -74.100,   4.650),
    ("Lima",               "South America",  -77.050, -12.050),
    ("Manaus_edge",        "South America",  -60.000,  -3.050),
    ("Buenos_Aires",       "South America",  -58.450, -34.600),
    ("Brisbane",           "Oceania",        153.020, -27.470),
    ("Auckland",           "Oceania",        174.760, -36.850),
]

areas = [
    {"name": name, "continent": continent, "lon": lon, "lat": lat, "split": "train"}
    for name, continent, lon, lat in new_locations
]

config_folder = Path("data/config")
config_folder.mkdir(parents=True, exist_ok=True)

print("Round-1 areas:", len(existing_areas), "| new training areas:", len(areas))

Round-1 areas: 34 | new training areas: 34


### Separation check (round-1 and new areas together)

Areas from different splits must be far apart, otherwise the test set contains almost the same
landscape as training. Areas must not overlap each other, and none may be near the case-study city (Mannheim).

In [4]:
def distance_km(a, b):
    lat1, lon1, lat2, lon2 = map(math.radians, [a["lat"], a["lon"], b["lat"], b["lon"]])
    h = (math.sin((lat2 - lat1) / 2) ** 2
         + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2)
    return 2 * 6371 * math.asin(math.sqrt(h))


CASE_STUDY = {"name": "Mannheim", "lon": 8.466, "lat": 49.487}
MIN_DISTANCE_KM = 50
overlap_km = 2 * math.sqrt(2) * BUFFER_METRES / 1000   # corner to corner of two squares

all_areas = existing_areas + areas
problems = []
for i, a in enumerate(all_areas):
    if distance_km(a, CASE_STUDY) < MIN_DISTANCE_KM:
        problems.append(f"{a['name']} is close to the case-study city")
    for b in all_areas[i + 1:]:
        d = distance_km(a, b)
        if d < overlap_km:
            problems.append(f"{a['name']} and {b['name']} may overlap ({d:.0f} km)")
        elif a["split"] != b["split"] and d < MIN_DISTANCE_KM:
            problems.append(f"{a['name']} ({a['split']}) and {b['name']} ({b['split']}) only {d:.0f} km apart")

print("\n".join(problems) if problems else "Separation check passed")

Separation check passed


# 3. Which satellite track covers each area in all four months?

A track is one orbit direction plus one relative orbit number (for example `ASCENDING_15`).
Using the same track in every month means the same viewing angle and the **same pixel grid**,
so the four months line up pixel by pixel.

Outside Europe there are often only 2–3 scenes per month on one track. That is enough for a median,
so these areas are kept (status `few_scenes`).

In [5]:
def add_track(image):
    direction = ee.String(image.get("orbitProperties_pass"))
    orbit = ee.Number(image.get("relativeOrbitNumber_start")).format("%d")
    return image.set("track", direction.cat("_").cat(orbit))


results = []
output_path = config_folder / f"candidate_availability_{PLAN_VERSION}.json"

for number, area in enumerate(areas, start=1):
    print(f"[{number}/{len(areas)}] Checking {area['name']}...", flush=True)

    result = dict(area)

    try:
        region = ee.Geometry.Point([area["lon"], area["lat"]]).buffer(BUFFER_METRES).bounds()

        collection = (
            ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(region)
            .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.eq("resolution_meters", 10))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
            .map(add_track)
        )

        # Fetch all four monthly histograms in one request per area
        queries = {}
        for month in MONTHS:
            start = ee.Date.fromYMD(YEAR, month, 1)
            queries[str(month)] = (
                collection
                .filterDate(start, start.advance(1, "month"))
                .aggregate_histogram("track")
            )

        counts = ee.Dictionary(queries).getInfo()

        common_tracks = set(counts[str(MONTHS[0])])
        for month in MONTHS[1:]:
            common_tracks.intersection_update(counts[str(month)])

        if not common_tracks:
            result["status"] = "no_common_track"
            print("  No single track available in all four months.")

        else:
            # Maximise the weakest month's count, then the total count
            best_track = max(
                sorted(common_tracks),
                key=lambda track: (
                    min(counts[str(m)][track] for m in MONTHS),
                    sum(counts[str(m)][track] for m in MONTHS)
                )
            )

            direction, orbit = best_track.rsplit("_", 1)
            scene_counts = [int(counts[str(m)][best_track]) for m in MONTHS]

            result.update({
                "direction": direction,
                "orbit": int(orbit),
                "scene_counts": scene_counts,
                "status": "candidate" if min(scene_counts) >= 3 else "few_scenes"
            })

            print(f"  {best_track} | Jan/Apr/Jul/Oct: {scene_counts} | {result['status']}")

    except Exception as error:
        result["status"] = "query_error"
        result["error"] = str(error)
        print("  Query failed:", error)

    results.append(result)

    # Save completed checks so their results are not lost
    output_path.write_text(
        json.dumps({
            "year": YEAR,
            "months": MONTHS,
            "buffer_metres": BUFFER_METRES,
            "areas": results
        }, indent=2),
        encoding="utf-8"
    )

print("\nSummary")
for status in ["candidate", "few_scenes", "no_common_track", "query_error"]:
    print(status, ":", sum(r["status"] == status for r in results))

print("\nResults saved to:", output_path)

[1/34] Checking Tabernas_desert...
  ASCENDING_1 | Jan/Apr/Jul/Oct: [9, 7, 7, 8] | candidate
[2/34] Checking Riyadh_edge...
  ASCENDING_72 | Jan/Apr/Jul/Oct: [3, 2, 3, 2] | few_scenes
[3/34] Checking Ouarzazate...
  DESCENDING_154 | Jan/Apr/Jul/Oct: [8, 7, 8, 7] | candidate
[4/34] Checking Phoenix_edge...
  DESCENDING_27 | Jan/Apr/Jul/Oct: [8, 7, 8, 8] | candidate
[5/34] Checking Antofagasta...
  ASCENDING_47 | Jan/Apr/Jul/Oct: [5, 5, 5, 5] | candidate
[6/34] Checking Alice_Springs...
  DESCENDING_104 | Jan/Apr/Jul/Oct: [3, 2, 3, 3] | few_scenes
[7/34] Checking Camargue...
  ASCENDING_161 | Jan/Apr/Jul/Oct: [6, 8, 7, 8] | candidate
[8/34] Checking Danube_delta...
  DESCENDING_36 | Jan/Apr/Jul/Oct: [7, 7, 5, 6] | candidate
[9/34] Checking Poyang_lake...
  ASCENDING_40 | Jan/Apr/Jul/Oct: [8, 7, 8, 9] | candidate
[10/34] Checking Okavango...
  ASCENDING_14 | Jan/Apr/Jul/Oct: [2, 3, 3, 2] | few_scenes
[11/34] Checking Everglades_edge...
  ASCENDING_48 | Jan/Apr/Jul/Oct: [2, 2, 3, 2] | few_

# 4. Coverage and class distribution

For each area: what share of the pixels has data in both VV and VH in each month,
and which WorldCover classes are present.
Areas with less than 90% coverage in any month are marked `low_coverage` and are not exported.
Move their coordinates a little (or pick another place) and run cells 3 and 4 again.

In [6]:
plan = json.loads(output_path.read_text(encoding="utf-8"))
year = plan["year"]
months = plan["months"]
buffer_metres = plan["buffer_metres"]

selected_areas = [
    area for area in plan["areas"]
    if area["status"] in ["candidate", "few_scenes"]
]

worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")

class_names = {
    10: "Trees", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare / sparse vegetation", 70: "Snow and ice",
    80: "Permanent water", 90: "Herbaceous wetland", 95: "Mangroves",
    100: "Moss and lichen"
}

MIN_COVERAGE = 90

print("Areas with a usable track:", len(selected_areas))

Areas with a usable track: 34


In [7]:
check_path = config_folder / f"coverage_and_classes_{PLAN_VERSION}.json"

# Resume earlier checks only when the input plan matches
if check_path.exists():
    saved = json.loads(check_path.read_text(encoding="utf-8"))
    if saved["plan"] != plan:
        raise ValueError(
            "The location plan has changed. Use a new PLAN_VERSION "
            "to keep the previous results separate."
        )
else:
    saved = {"plan": plan, "checks": {}}

for index, area in enumerate(selected_areas, start=1):
    name = area["name"]
    previous = saved["checks"].get(name, {})

    if previous.get("status") in ["complete", "low_coverage"]:
        print(f"[{index}/{len(selected_areas)}] {name}: already checked")
        continue

    print(f"[{index}/{len(selected_areas)}] Checking {name}...", flush=True)

    try:
        region = ee.Geometry.Point([area["lon"], area["lat"]]).buffer(buffer_metres).bounds()

        # Choose the local UTM coordinate system, measured in metres
        zone = min(60, max(1, math.floor((area["lon"] + 180) / 6) + 1))
        epsg = (32600 if area["lat"] >= 0 else 32700) + zone
        crs = f"EPSG:{epsg}"

        collection = (
            ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(region)
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.eq("resolution_meters", 10))
            .filter(ee.Filter.eq("orbitProperties_pass", area["direction"]))
            .filter(ee.Filter.eq("relativeOrbitNumber_start", area["orbit"]))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
            .select(["VV", "VH"])
        )

        coverage_bands = []
        for month in months:
            start = ee.Date.fromYMD(year, month, 1)
            monthly = collection.filterDate(start, start.advance(1, "month"))

            # A pixel is usable only when BOTH channels have data
            usable = (
                monthly.count()
                .unmask(0, sameFootprint=False)
                .reduce(ee.Reducer.min())
                .gt(0)
                .rename(f"month_{month:02d}")
            )
            coverage_bands.append(usable)

        coverage_stats = ee.Image.cat(*coverage_bands).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=region, crs=crs,
            scale=10, maxPixels=10_000_000, tileScale=4
        )

        label_counts = worldcover.reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(), geometry=region, crs=crs,
            scale=10, maxPixels=10_000_000, tileScale=4
        ).get("Map")

        stats = ee.Dictionary({
            "coverage": coverage_stats,
            "label_counts": label_counts
        }).getInfo()

        coverage_percent = {key: round(value * 100, 3) for key, value in stats["coverage"].items()}
        counts = {int(float(code)): count for code, count in stats["label_counts"].items()}

        total = sum(counts.values())
        if total == 0:
            raise ValueError("No valid WorldCover labels found.")

        class_percent = {
            class_names.get(code, f"Unknown code {code}"): round(100 * count / total, 4)
            for code, count in counts.items()
        }

        status = "complete" if min(coverage_percent.values()) >= MIN_COVERAGE else "low_coverage"

        saved["checks"][name] = {
            "status": status,
            "split": area["split"],
            "crs": crs,
            "coverage_percent": coverage_percent,
            "class_percent": class_percent,
            "label_counts": counts
        }

        coverage_text = ", ".join(
            f"{month:02d}: {coverage_percent[f'month_{month:02d}']:.2f}%" for month in months
        )
        print("  Coverage:", coverage_text, "" if status == "complete" else "-> LOW COVERAGE")

        largest = sorted(class_percent.items(), key=lambda item: item[1], reverse=True)[:4]
        print("  Main classes:", ", ".join(f"{label} {share:.1f}%" for label, share in largest))

    except Exception as error:
        saved["checks"][name] = {"status": "error", "error": str(error)}
        print("  Check failed:", error)

    # Save after every location
    temporary_path = check_path.with_suffix(".tmp")
    temporary_path.write_text(json.dumps(saved, indent=2), encoding="utf-8")
    temporary_path.replace(check_path)

for status in ["complete", "low_coverage", "error"]:
    print(status, ":", sum(item["status"] == status for item in saved["checks"].values()))
print("Saved results:", check_path)

[1/34] Tabernas_desert: already checked
[2/34] Riyadh_edge: already checked
[3/34] Ouarzazate: already checked
[4/34] Phoenix_edge: already checked
[5/34] Antofagasta: already checked
[6/34] Alice_Springs: already checked
[7/34] Camargue: already checked
[8/34] Danube_delta: already checked
[9/34] Poyang_lake: already checked
[10/34] Okavango: already checked
[11/34] Everglades_edge: already checked
[12/34] Pantanal: already checked
[13/34] Dublin: already checked
[14/34] Po_valley: already checked
[15/34] Jyvaskyla_forest: already checked
[16/34] Andalusia_farmland: already checked
[17/34] Jakarta: already checked
[18/34] Dhaka: already checked
[19/34] Tehran: already checked
[20/34] Astana_steppe: already checked
[21/34] Mekong_delta: already checked
[22/34] Addis_Ababa: already checked
[23/34] Kano_rural: already checked
[24/34] Kinshasa: already checked
[25/34] Nile_delta: already checked
[26/34] Iowa_farmland: already checked
[27/34] Alberta_prairie: already checked
[28/34] Guadal

### Class distribution per split

Pixel counts of all areas together, per split. This table goes into the report (class distribution).

In [8]:
for split in ["train", "validation", "test"]:
    totals = {}
    for item in saved["checks"].values():
        if item["status"] != "complete" or item["split"] != split:
            continue
        for code, count in item["label_counts"].items():
            totals[int(code)] = totals.get(int(code), 0) + count

    all_pixels = sum(totals.values())
    print(f"\n{split} ({all_pixels:,} pixels)")
    for code, name in class_names.items():
        print(f"  {name:<26}{100 * totals.get(code, 0) / max(all_pixels, 1):6.2f}%")


train (80,094,520.3372549 pixels)
  Trees                      14.91%
  Shrubland                   4.60%
  Grassland                  10.20%
  Cropland                   11.83%
  Built-up                   30.24%
  Bare / sparse vegetation    8.88%
  Snow and ice                0.00%
  Permanent water             8.29%
  Herbaceous wetland         11.01%
  Mangroves                   0.04%
  Moss and lichen             0.00%

validation (0 pixels)
  Trees                       0.00%
  Shrubland                   0.00%
  Grassland                   0.00%
  Cropland                    0.00%
  Built-up                    0.00%
  Bare / sparse vegetation    0.00%
  Snow and ice                0.00%
  Permanent water             0.00%
  Herbaceous wetland          0.00%
  Mangroves                   0.00%
  Moss and lichen             0.00%

test (0 pixels)
  Trees                       0.00%
  Shrubland                   0.00%
  Grassland                   0.00%
  Cropland               

# 5. Export radar and labels

For every area with full coverage: four monthly radar images (VV, VH) and one WorldCover image,
all on the pixel grid of the chosen track (same idea as in the pilot notebook).

Files are stored per plan version (`data/raw/v2/<area>/`). Each area folder has an `info.json`
with the track and the pixel grid. An existing file is only reused when its coordinate system, grid,
size and band count match that `info.json`, and the `info.json` matches the current plan.
If the settings of an area changed, its old files are deleted and downloaded again.

In [9]:
NODATA = -9999
PIXEL_SIZE = 10
raw_folder = Path("data/raw") / PLAN_VERSION
raw_folder.mkdir(parents=True, exist_ok=True)

# settings that define an area's files; if any of them changes, the files must be downloaded again
IDENTITY_KEYS = ["lon", "lat", "direction", "orbit", "crs", "width", "height"]

export_areas = [
    area for area in selected_areas
    if saved["checks"].get(area["name"], {}).get("status") == "complete"
]
print("Areas to export:", len(export_areas))


def track_collection(area, region):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(region)
        .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))
        .filter(ee.Filter.eq("orbitProperties_pass", area["direction"]))
        .filter(ee.Filter.eq("relativeOrbitNumber_start", area["orbit"]))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH"])
    )


def snapped_grid(region, collection):
    # Same as the pilot: snap the area to the original radar pixel grid
    radar_projection = ee.Image(collection.first()).select("VV").projection().getInfo()

    crs = radar_projection["crs"]
    origin_x = radar_projection["transform"][2]
    origin_y = radar_projection["transform"][5]

    bounds = region.bounds(maxError=1, proj=ee.Projection(crs)).coordinates().getInfo()[0]
    xs = [point[0] for point in bounds]
    ys = [point[1] for point in bounds]

    left_col = math.floor((min(xs) - origin_x) / PIXEL_SIZE)
    right_col = math.ceil((max(xs) - origin_x) / PIXEL_SIZE)
    top_row = math.floor((origin_y - max(ys)) / PIXEL_SIZE)
    bottom_row = math.ceil((origin_y - min(ys)) / PIXEL_SIZE)

    width = right_col - left_col
    height = bottom_row - top_row
    left = origin_x + left_col * PIXEL_SIZE
    top = origin_y - top_row * PIXEL_SIZE

    return crs, [PIXEL_SIZE, 0, left, 0, -PIXEL_SIZE, top], width, height


def download_tif(image, path, crs, grid_transform, width, height):
    url = image.getDownloadURL({
        "crs": crs,
        "crs_transform": grid_transform,
        "dimensions": [width, height],
        "format": "GEO_TIFF"
    })
    response = requests.get(url, timeout=300)
    response.raise_for_status()
    path.write_bytes(response.content)

    with rasterio.open(path, "r+") as dataset:
        dataset.nodata = NODATA


def check_raster(path, info, band_count):
    # returns "ok" or the reason why the file does not match info.json
    if not path.exists():
        return "missing"
    try:
        with rasterio.open(path) as src:
            if src.crs is None or src.crs.to_string() != info["crs"]:
                return "wrong coordinate system"
            if (src.width, src.height) != (info["width"], info["height"]):
                return "wrong size"
            # rtol=0: the default relative tolerance would accept shifts of ~50 m on UTM coordinates
            if not np.allclose(tuple(src.transform)[:6], info["transform"], rtol=0, atol=1e-6):
                return "wrong pixel grid"
            if src.count != band_count:
                return f"{src.count} bands instead of {band_count}"
    except rasterio.errors.RasterioIOError:
        return "cannot be opened"
    return "ok"


def same_settings(old_info, new_info):
    return (all(old_info.get(key) == new_info[key] for key in IDENTITY_KEYS)
            and np.allclose(old_info.get("transform", [0] * 6), new_info["transform"], rtol=0, atol=1e-6))

Areas to export: 34


In [10]:
for index, area in enumerate(export_areas, start=1):
    name = area["name"]
    area_folder = raw_folder / name
    area_folder.mkdir(parents=True, exist_ok=True)

    info_path = area_folder / "info.json"
    label_path = area_folder / f"worldcover_{year}.tif"
    month_paths = {month: area_folder / f"s1_{year}_{month:02d}.tif" for month in months}

    print(f"[{index}/{len(export_areas)}] {name}", flush=True)

    try:
        region = ee.Geometry.Point([area["lon"], area["lat"]]).buffer(buffer_metres).bounds()
        collection = track_collection(area, region)
        crs, grid_transform, width, height = snapped_grid(region, collection)

        info = {
            **area, "year": year, "months": months, "buffer_metres": buffer_metres,
            "crs": crs, "transform": grid_transform, "width": width, "height": height
        }

        # files from other settings (or without info.json) cannot be trusted
        old_info = json.loads(info_path.read_text()) if info_path.exists() else None
        if old_info is None or not same_settings(old_info, info):
            for old_file in area_folder.glob("*.tif"):
                old_file.unlink()
                print("  deleted outdated", old_file.name)
        info_path.write_text(json.dumps(info, indent=2), encoding="utf-8")

        for month, path in month_paths.items():
            if check_raster(path, info, 2) == "ok":
                continue
            start = ee.Date.fromYMD(year, month, 1)
            monthly = collection.filterDate(start, start.advance(1, "month")).median()
            download_tif(monthly.toFloat().unmask(NODATA, sameFootprint=False),
                         path, crs, grid_transform, width, height)
            status = check_raster(path, info, 2)
            if status != "ok":
                raise ValueError(f"{path.name}: {status}")
            print(f"  month {month:02d} saved")

        if check_raster(label_path, info, 1) != "ok":
            download_tif(worldcover.toInt16().unmask(NODATA, sameFootprint=False),
                         label_path, crs, grid_transform, width, height)
            status = check_raster(label_path, info, 1)
            if status != "ok":
                raise ValueError(f"{label_path.name}: {status}")
            print("  labels saved")

        print("  all files ok")

    except Exception as error:
        print("  Export failed:", error)

[1/34] Tabernas_desert
  all files ok
[2/34] Riyadh_edge
  all files ok
[3/34] Ouarzazate
  all files ok
[4/34] Phoenix_edge
  all files ok
[5/34] Antofagasta
  all files ok
[6/34] Alice_Springs
  all files ok
[7/34] Camargue
  all files ok
[8/34] Danube_delta
  all files ok
[9/34] Poyang_lake
  all files ok
[10/34] Okavango
  all files ok
[11/34] Everglades_edge
  all files ok
[12/34] Pantanal
  all files ok
[13/34] Dublin
  all files ok
[14/34] Po_valley
  all files ok
[15/34] Jyvaskyla_forest
  all files ok
[16/34] Andalusia_farmland
  all files ok
[17/34] Jakarta
  all files ok
[18/34] Dhaka
  all files ok
[19/34] Tehran
  all files ok
[20/34] Astana_steppe
  all files ok
[21/34] Mekong_delta
  all files ok
[22/34] Addis_Ababa
  all files ok
[23/34] Kano_rural
  all files ok
[24/34] Kinshasa
  all files ok
[25/34] Nile_delta
  all files ok
[26/34] Iowa_farmland
  all files ok
[27/34] Alberta_prairie
  all files ok
[28/34] Guadalajara
  all files ok
[29/34] Bogota
  all files ok
[30

# 6. Patches

Each area is cut into non-overlapping 256 × 256 patches. Every month is a separate sample with
the **same** WorldCover label, so the model learns that a field is still cropland in winter.

WorldCover's 11 source classes are mapped to 7 output classes:

| Output class | WorldCover source codes | Note |
|---|---|---|
| 0 Trees (incl. mangroves) | 10 Tree cover, 95 Mangroves | mangroves are rare and look like forest to radar |
| 1 Grassland / shrubland | 30 Grassland, 20 Shrubland, 100 Moss and lichen | radar separates these poorly |
| 2 Cropland | 40 Cropland | |
| 3 Built-up | 50 Built-up | |
| 4 Bare / sparse vegetation | 60 Bare / sparse vegetation | |
| 5 Permanent water | 80 Permanent water bodies | |
| 6 Herbaceous wetland | 90 Herbaceous wetland | mangroves are **not** included here |
| 255 ignored | 70 Snow and ice, no data | snow is out of scope (about 0.01% of training pixels) |

255 means "no label": these pixels are excluded from the loss and from all metrics in the training notebook.
The model has no snow class, so results for snow-covered areas are not claimed.

Before cutting, every area's five files are checked against its `info.json`
(same coordinate system, grid, size; 2 radar bands, 1 label band).
Patches with any missing or non-finite radar value, or with less than 90% labelled pixels, are skipped.

In [11]:
PATCH_SIZE = 256
IGNORE = 255

CLASSES = ["Trees (incl. mangroves)", "Grassland / shrubland", "Cropland", "Built-up",
           "Bare / sparse vegetation", "Permanent water", "Herbaceous wetland"]

# 70 Snow and ice is left out on purpose -> IGNORE
REMAP = {10: 0, 95: 0, 30: 1, 20: 1, 100: 1, 40: 2, 50: 3, 60: 4, 80: 5, 90: 6}

lookup = np.full(101, IGNORE, dtype=np.uint8)
for code, new_class in REMAP.items():
    lookup[code] = new_class


def to_classes(codes):
    classes = np.full(codes.shape, IGNORE, dtype=np.uint8)
    known = (codes >= 0) & (codes <= 100)
    classes[known] = lookup[codes[known]]
    return classes

In [12]:
patch_folder = Path(f"data/patches_{PLAN_VERSION}")
patch_folder.mkdir(parents=True, exist_ok=True)

images = {"train": [], "validation": [], "test": []}
masks = {"train": [], "validation": [], "test": []}
meta = {"train": [], "validation": [], "test": []}

for area in export_areas:
    name = area["name"]
    split = area["split"]
    area_folder = raw_folder / name
    info_path = area_folder / "info.json"
    label_path = area_folder / f"worldcover_{year}.tif"
    month_paths = {month: area_folder / f"s1_{year}_{month:02d}.tif" for month in months}

    if not info_path.exists():
        print(f"{name:<18} skipped: no info.json")
        continue
    info = json.loads(info_path.read_text())

    # all five files must match info.json, so they also match each other
    problems = {label_path.name: check_raster(label_path, info, 1)}
    for path in month_paths.values():
        problems[path.name] = check_raster(path, info, 2)
    problems = {file: status for file, status in problems.items() if status != "ok"}
    if problems:
        print(f"{name:<18} skipped:", problems)
        continue

    with rasterio.open(label_path) as src:
        labels = to_classes(src.read(1))

    kept = 0
    for month, path in month_paths.items():
        with rasterio.open(path) as src:
            radar = src.read()

        height, width = labels.shape
        for row in range(0, height - PATCH_SIZE + 1, PATCH_SIZE):
            for col in range(0, width - PATCH_SIZE + 1, PATCH_SIZE):
                x = radar[:, row:row + PATCH_SIZE, col:col + PATCH_SIZE]
                y = labels[row:row + PATCH_SIZE, col:col + PATCH_SIZE]

                if (x == NODATA).any() or not np.isfinite(x).all():
                    continue
                if (y != IGNORE).mean() < 0.9:
                    continue

                images[split].append(np.moveaxis(x, 0, -1).astype(np.float16))
                masks[split].append(y)
                meta[split].append(f"{name},{area['continent']},{month},{row},{col}")
                kept += 1

    print(f"{name:<18} {split:<11} patches: {kept}")

Tabernas_desert    train       patches: 144
Riyadh_edge        train       patches: 144
Ouarzazate         train       patches: 144
Phoenix_edge       train       patches: 144
Antofagasta        train       patches: 144
Alice_Springs      train       patches: 144
Camargue           train       patches: 144
Danube_delta       train       patches: 144
Poyang_lake        train       patches: 144
Okavango           train       patches: 144
Everglades_edge    train       patches: 144
Pantanal           train       patches: 94
Dublin             train       patches: 144
Po_valley          train       patches: 144
Jyvaskyla_forest   train       patches: 144
Andalusia_farmland train       patches: 144
Jakarta            train       patches: 144
Dhaka              train       patches: 144
Tehran             train       patches: 144
Astana_steppe      train       patches: 144
Mekong_delta       train       patches: 144
Addis_Ababa        train       patches: 100
Kano_rural         train       pa

In [13]:
for split in ["train", "validation", "test"]:
    if not images[split]:
        continue                      # round 2 only adds training areas
    x = np.stack(images[split])
    y = np.stack(masks[split])
    np.save(patch_folder / f"{split}_x.npy", x)
    np.save(patch_folder / f"{split}_y.npy", y)
    (patch_folder / f"{split}_meta.csv").write_text(
        "area,continent,month,row,col\n" + "\n".join(meta[split]), encoding="utf-8"
    )

    labelled = y[y != IGNORE]
    print(f"\n{split}: x {x.shape}, y {y.shape}")
    for i, name in enumerate(CLASSES):
        print(f"  {name:<14}{100 * np.mean(labelled == i):6.2f}%")

(patch_folder / "classes.json").write_text(json.dumps({
    "classes": CLASSES,
    "worldcover_to_class": {str(code): new for code, new in REMAP.items()},
    "ignore": IGNORE,
    "out_of_scope": {"70": "Snow and ice"},
    "plan_version": PLAN_VERSION,
}, indent=2))


train: x (4618, 256, 256, 2), y (4618, 256, 256)
  Trees (incl. mangroves) 14.48%
  Grassland / shrubland 15.24%
  Cropland       12.25%
  Built-up       29.33%
  Bare / sparse vegetation  9.34%
  Permanent water  8.31%
  Herbaceous wetland 11.05%


453

# 7. What did the new areas really add?

A place called "wetland" may still be mostly water or trees in WorldCover. This table shows, per new area,
how many patches survived the filters and how much herbaceous wetland and bare land they really contain.
It reads the saved files, so it can be run again at any time.

In [14]:
y_new = np.load(patch_folder / "train_y.npy", mmap_mode="r")
rows = [r.split(",") for r in (patch_folder / "train_meta.csv").read_text().strip().split("\n")[1:]]
area_names = np.array([r[0] for r in rows])

WETLAND, BARE, WATER, BUILT = 6, 4, 5, 3
shares = {}
print(f"{'area':<20}{'patches':>8}{'wetland %':>11}{'bare %':>8}{'water %':>9}{'built %':>9}")
for name in sorted(set(area_names)):
    labels = np.asarray(y_new[np.flatnonzero(area_names == name)])
    labels = labels[labels != IGNORE]
    shares[name] = {k: 100 * np.mean(labels == k) for k in (WETLAND, BARE, WATER, BUILT)}
    print(f"{name:<20}{np.sum(area_names == name):8d}{shares[name][WETLAND]:11.2f}{shares[name][BARE]:8.2f}"
          f"{shares[name][WATER]:9.2f}{shares[name][BUILT]:9.2f}")

print("\nareas with more than 5% herbaceous wetland:", sum(s[WETLAND] > 5 for s in shares.values()))
print("areas with more than 5% bare / sparse vegetation:", sum(s[BARE] > 5 for s in shares.values()))

area                 patches  wetland %  bare %  water %  built %
Addis_Ababa              100       0.00    2.90     0.01    57.03
Alberta_prairie          144       0.01    0.10     1.22     0.19
Alice_Springs            144       0.00    1.77     0.10     5.27
Andalusia_farmland       144       0.00    1.69     0.04     2.24
Antofagasta              144       0.00   44.07    46.03     9.71
Astana_steppe            144       0.83    6.58     3.13    41.09
Auckland                 128       0.23    0.27    29.00    36.12
Bogota                   100       0.05    0.69     0.45    78.18
Brisbane                 120       0.03    0.22     4.55    51.39
Buenos_Aires             144       0.19    0.24    10.56    77.11
Camargue                 144      23.92    1.84    41.39     0.32
Danube_delta             144      68.09    0.37    19.83     0.15
Dhaka                    144       1.30    3.47     4.25    46.56
Dublin                   144       0.23    0.47    10.35    45.67
Everglades

In [16]:
import numpy as np
from pathlib import Path
old, new = Path("../patches_v3_extra_backup"), Path("data/patches_v3_extra")
for f in ["train_x.npy", "train_y.npy"]:
    a, b = np.load(old / f, mmap_mode="r"), np.load(new / f, mmap_mode="r")
    print(f, a.shape, b.shape, "identical:", a.shape == b.shape and np.array_equal(a, b))
print("meta identical:", (old / "train_meta.csv").read_text() == (new / "train_meta.csv").read_text())

train_x.npy (4618, 256, 256, 2) (4618, 256, 256, 2) identical: True
train_y.npy (4618, 256, 256) (4618, 256, 256) identical: True
meta identical: True
